# Curve-based fusion detection: split roles and gate check

Prepares and checks the own approach (Branch B: a human-vs-ai window detector whose p(AI) curves feed a stacker together with Branch A, the tuned 3-class XLM-R from `hw3-encoder-tuning.ipynb`).

1. **Split roles**: which training texts train the detector and which train the stacker, with leakage checks. Written to `splits/curve_roles.csv` for the cluster jobs.
2. **Reference and error anatomy**: Branch A's validation score (R0) and where its human ↔ fusion errors are (length, domain).
3. **Twin boundaries**: for validation fusion texts with a human twin, the token where the copied human part ends.
4. **Gate texts**: the texts the gate job scores, written to `splits/gate_texts.csv`.

Runs on CPU. Only the tokenizer is used here; all models run on the cluster.

In [1]:
import os
import re
import sys
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from transformers import AutoTokenizer

sys.path.insert(0, str(Path("../cluster").resolve()))
from encoder.data import (GATE_CSV, LABELS, N_TRAIN_SAMPLE, ROLES_CSV, SEED, build_roles, load_data, prefix_key,
                          stratified_sample)

COMPETITION = "ai-generated-text-detection-hw3-ucu-ai-2026"
BRANCH_A_DIR = Path("../outputs/branch_a/encoder")  # Kaggle output of hw3-encoder-tuning (probabilities only)
TOKENIZER = "xlm-roberta-base"                       # = FacebookAI/xlm-roberta-base, Branch A's tokenizer
BRANCH_A_TRAIN_WINDOWS = 104_407                     # "train_windows" in run_xlm-roberta-base.json

data = load_data(kagglehub.competition_download(COMPETITION))
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)
pd.DataFrame({"texts": {name: len(df) for name, df in data.items()}}).T

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


,train,val,test
texts,245116,27236,25000


## 1. Split roles
- **detector**: the human and ai texts of the fixed 50k training sample (fusion excluded). Branch A was trained on the whole 50k sample.
- **stacker**: 30k texts from the training folds that are *not* in the 50k sample and whose 200-character prefix matches no sample text. The 50k sample was drawn without grouping, so without this twin guard a stacker fusion text could have its human twin among the detector's training texts. Drawn with the label × language shares of the training folds.
- **val** (fold 0) and **test** are only scored.

The checks below must all be 0. The last cell checks that the 50k sample rebuilt here is the one Branch A was trained on: it must give the same number of training windows as the Kaggle run.

In [2]:
def split_sets(roles, data):
    """Texts of each role, plus val and test."""
    train = data["train"]
    sets = {role: train[train["id"].isin(ids)].reset_index(drop=True) for role, ids in roles.groupby("role")["id"]}
    return {**sets, "val": data["val"], "test": data["test"]}


def leakage_checks(sets, train_sample):
    """Number of violations of each no-leakage rule (all must be 0)."""
    sample_ids, sample_prefixes = set(train_sample["id"]), set(prefix_key(train_sample["text"]))
    ids = {name: set(df["id"]) for name, df in sets.items()}
    pure_sample_ids = set(train_sample.loc[train_sample["label"] != "fusion", "id"])
    return pd.Series({
        "detector ∩ stacker": len(ids["detector"] & ids["stacker"]),
        "val ∩ (detector ∪ stacker)": len(ids["val"] & (ids["detector"] | ids["stacker"])),
        "stacker ∩ 50k sample": len(ids["stacker"] & sample_ids),
        "stacker prefix in 50k sample": prefix_key(sets["stacker"]["text"]).isin(sample_prefixes).sum(),
        "val prefix in 50k sample": prefix_key(sets["val"]["text"]).isin(sample_prefixes).sum(),
        "test prefix in 50k sample": prefix_key(sets["test"]["text"]).isin(sample_prefixes).sum(),
        "detector ≠ human/ai of 50k sample": len(ids["detector"] ^ pure_sample_ids),
    }, name="violations")


roles = build_roles(data["train"])
sets = split_sets(roles, data)
train_sample = stratified_sample(data["train"], N_TRAIN_SAMPLE)
checks = leakage_checks(sets, train_sample)
assert (checks == 0).all(), checks
checks.to_frame()

,violations
detector ∩ stacker,0
val ∩ (detector ∪ stacker),0
stacker ∩ 50k sample,0
stacker prefix in 50k sample,0
val prefix in 50k sample,0
test prefix in 50k sample,0
detector ≠ human/ai of 50k sample,0


In [3]:
def label_mix(sets):
    """Texts and label shares (%) of each labelled set."""
    rows = {name: {"texts": len(df), **(100 * df["label"].value_counts(normalize=True)).to_dict()}
            for name, df in sets.items() if "label" in df}
    return pd.DataFrame(rows).T[["texts", *LABELS]].fillna(0)


label_mix(sets).round(1)

,texts,human,ai,fusion
detector,34638.0,35.4,64.6,0.0
stacker,29999.0,24.5,44.8,30.7
val,27236.0,24.5,44.8,30.7


In [4]:
def branch_a_train_windows(texts, max_windows=4, batch_size=2_000):
    """Training windows Branch A got from these texts (512 tokens, overlap 128, at most 4 per text)."""
    total = 0
    for start in range(0, len(texts), batch_size):
        enc = tokenizer(list(texts[start:start + batch_size]), truncation=True, max_length=512, stride=128,
                        return_overflowing_tokens=True)
        total += np.minimum(np.bincount(enc["overflow_to_sample_mapping"]), max_windows).sum()
    return int(total)


n_windows = branch_a_train_windows(train_sample["text"].tolist())
assert n_windows == BRANCH_A_TRAIN_WINDOWS, n_windows
roles.to_csv(ROLES_CSV, index=False)
print(f"50k sample reproduces Branch A's {n_windows:,} training windows; wrote {ROLES_CSV.name} ({len(roles):,} rows)")

50k sample reproduces Branch A's 104,407 training windows; wrote curve_roles.csv (64,637 rows)


## 2. Reference (R0) and where Branch A fails
R0 is Branch A's validation macro-F1 from its saved probabilities (mean over 512-token windows, argmax). The tables show Branch A's error rate on human and fusion texts by token length, and where the human ↔ fusion errors are. "pair error" = a human text predicted as fusion or a fusion text predicted as human.

In [5]:
LENGTH_BINS = [0, 128, 256, 512, 1024, np.inf]
LENGTH_LABELS = ["<128", "128-256", "256-512", "512-1024", ">1024"]


def score(y_true, y_pred):
    """Macro-F1 and per-class F1."""
    per_class = f1_score(y_true, y_pred, labels=LABELS, average=None)
    return {"macro_f1": f1_score(y_true, y_pred, average="macro"),
            **{f"f1_{label}": f1 for label, f1 in zip(LABELS, per_class)}}


def with_branch_a(df, path):
    """df with Branch A's text probabilities, its argmax prediction and the error flags."""
    out = df.merge(pd.read_csv(path), on="id", validate="1:1")
    out["pred"] = np.array(LABELS)[out[LABELS].to_numpy().argmax(axis=1)]
    out["error"] = out["pred"] != out["label"]
    out["pair_error"] = out["error"] & out["label"].isin(["human", "fusion"]) & out["pred"].isin(["human", "fusion"])
    return out


def token_counts(texts):
    """Number of XLM-R tokens of each text (without special tokens)."""
    return [len(ids) for ids in tokenizer(list(texts), add_special_tokens=False, verbose=False)["input_ids"]]


val = with_branch_a(data["val"], BRANCH_A_DIR / "val_probs_xlm-roberta-base.csv")
val["n_tokens"] = token_counts(val["text"])
val["length"] = pd.cut(val["n_tokens"], LENGTH_BINS, labels=LENGTH_LABELS)
pd.DataFrame([score(val["label"], val["pred"])], index=["R0: Branch A, mean"]).round(4)

,macro_f1,f1_human,f1_ai,f1_fusion
"R0: Branch A, mean",0.902,0.8362,0.9901,0.8797


In [6]:
def error_rates(df, by):
    """Error and pair-error rates (%) of human and fusion texts per value of `by`."""
    pair = df[df["label"].isin(["human", "fusion"])]
    return (pair.groupby(["label", by], observed=True)
                .agg(texts=("error", "size"), error_pct=("error", "mean"), pair_error_pct=("pair_error", "mean"))
                .assign(error_pct=lambda d: 100 * d["error_pct"], pair_error_pct=lambda d: 100 * d["pair_error_pct"]))


error_rates(val, "length").round(1)

texts  error_pct  pair_error_pct
label  length                                    
fusion <128        507       17.2             9.1
       128-256     831       15.2            11.7
       256-512    2132        9.6             8.7
       512-1024   2938        5.0             3.9
       >1024      1960        3.8             3.2
human  <128        581       20.3            13.4
       128-256     599       23.9            22.2
       256-512    1417       16.7            16.1
       512-1024   2351       18.9            18.6
       >1024      1725       32.9            32.6

In [7]:
pair_errors = val[val["pair_error"]]
pd.concat({"by length": 100 * pair_errors["length"].value_counts(normalize=True).sort_index(),
           "by domain": 100 * pair_errors["domain"].value_counts(normalize=True)}).rename(
    f"% of the {len(pair_errors):,} human ↔ fusion errors").to_frame().round(1)

% of the 1,947 human ↔ fusion errors
by length <128                                                     6.4
          128-256                                                 11.8
          256-512                                                 21.2
          512-1024                                                28.5
          >1024                                                   32.2
by domain blog_authorship_corpus                                  24.8
          euro_court_cases                                        20.6
          bundestag                                               20.0
          student_essays                                          18.7
          house_of_commons                                         8.4
          cnn_news                                                 4.2
          arxiv_papers                                             2.4
          gutenberg                                                1.0

## 3. Twin boundaries
A fusion text with a human twin (same 200-character prefix, see EDA section 5) copies the twin word for word up to some point, then the AI-written part starts. The boundary is that point, as a token index in the fusion text and as a share of its length. Twins share a fold, so validation fusion texts find their twin within validation. Boundaries are only used to *evaluate* curves (gate criterion K1, xAI), never for fitting.

In [8]:
def boundary_char(fusion_text, human_text):
    """Character offset where the fusion text stops copying its human twin (end of the last copied word)."""
    copied = len(os.path.commonprefix([fusion_text.split(), human_text.split()]))
    word_ends = [m.end() for m in re.finditer(r"\S+", fusion_text)]
    return word_ends[copied - 1] if copied else 0


def twin_boundaries(fusion, humans):
    """Boundary token and share for fusion texts whose prefix matches exactly one human text in `humans`."""
    human_by_prefix = (humans.assign(prefix=prefix_key(humans["text"]))
                             .drop_duplicates("prefix", keep=False).set_index("prefix")["text"])
    fusion = fusion.assign(prefix=prefix_key(fusion["text"]))
    fusion = fusion[fusion["prefix"].isin(human_by_prefix.index)]
    chars = [boundary_char(f, human_by_prefix[p]) for f, p in zip(fusion["text"], fusion["prefix"])]
    enc = tokenizer(fusion["text"].tolist(), add_special_tokens=False, return_offsets_mapping=True, verbose=False)
    tokens = [int(np.searchsorted([start for start, _ in offsets], c)) for offsets, c in zip(enc["offset_mapping"], chars)]
    n_tokens = np.array([len(ids) for ids in enc["input_ids"]])
    return pd.DataFrame({"id": fusion["id"].to_numpy(), "boundary_char": chars, "boundary_tok": tokens,
                         "boundary_share": np.array(tokens) / n_tokens})


val_boundaries = twin_boundaries(val[val["label"] == "fusion"], val[val["label"] == "human"])
print(f"validation fusion texts with a twin boundary: {len(val_boundaries):,} of {(val['label'] == 'fusion').sum():,}")
val_boundaries[["boundary_tok", "boundary_share"]].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).T.round(2)

validation fusion texts with a twin boundary: 6,051 of 8,368


,0.10,0.25,0.50,0.75,0.90
boundary_tok,83.00,136.00,254.00,454.5,740.00
boundary_share,0.16,0.28,0.48,0.7,0.86


In [9]:
# Branch A misses more fusion texts when most of the text is copied human text.
missed = val.merge(val_boundaries, on="id").assign(missed=lambda d: d["pred"] != "fusion")
missed.groupby(pd.cut(missed["boundary_share"], [0, 0.25, 0.5, 0.75, 0.9, 1.0], include_lowest=True),
               observed=True)["missed"].agg(texts="size", missed_pct=lambda s: 100 * s.mean()).round(1)

,texts,missed_pct
boundary_share,,
"(-0.001, 0.25]",1269,5.4
"(0.25, 0.5]",1863,5.5
"(0.5, 0.75]",1728,5.6
"(0.75, 0.9]",785,10.1
"(0.9, 1.0]",406,23.6


## 4. Gate texts
The gate job scores two groups of texts:
- **val_plot**: 20 human and 20 fusion validation texts of at least 256 tokens (so a 128-token curve has several points), for the curve plots only. Each class is half Branch A errors, half correct. The fusion texts all have a known twin boundary.
- **stacker_slice**: 1,000 human, 1,000 fusion and 500 ai texts of the stacker set. The numeric gate criteria (K1–K3) are computed on these, not on validation.

In [10]:
PLOT_PER_GROUP = 10
PLOT_MIN_TOKENS = 256
SLICE_SIZES = {"human": 1_000, "fusion": 1_000, "ai": 500}


def plot_texts(val, boundaries):
    """10 texts each: human predicted fusion / human, fusion (with boundary) predicted human / fusion."""
    long = val[val["n_tokens"] >= PLOT_MIN_TOKENS]
    groups = {("human", "fusion"): long[long["label"] == "human"], ("human", "human"): long[long["label"] == "human"],
              ("fusion", "human"): long[(long["label"] == "fusion") & long["id"].isin(boundaries["id"])],
              ("fusion", "fusion"): long[(long["label"] == "fusion") & long["id"].isin(boundaries["id"])]}
    picks = [df[df["pred"] == pred].sample(PLOT_PER_GROUP, random_state=SEED) for (_, pred), df in groups.items()]
    return pd.concat(picks)


def stacker_slice(stacker):
    """Fixed random slice of the stacker set with SLICE_SIZES texts per label."""
    return pd.concat([stacker[stacker["label"] == label].sample(n, random_state=SEED) for label, n in SLICE_SIZES.items()])


gate = pd.concat([plot_texts(val, val_boundaries)[["id", "label", "pred"]].assign(set="val_plot"),
                  stacker_slice(sets["stacker"])[["id", "label"]].assign(set="stacker_slice")], ignore_index=True)
assert gate["id"].is_unique
gate[["id", "set"]].to_csv(GATE_CSV, index=False)
gate.groupby(["set", "label"]).size().rename("texts").to_frame()

texts
set           label        
stacker_slice ai        500
              fusion   1000
              human    1000
val_plot      fusion     20
              human      20